# Keras

A self-contained refresher on **Keras 3** — the high-level, multi-backend deep-learning API.

**Domain:** AI/ML Tooling  ·  **runnable:** yes

## 1. What & Why

**Keras is the batteries-included API for building, training, and shipping neural networks.** You describe a model as a stack (or graph) of layers, call `.compile()` to attach a loss/optimizer/metrics, then `.fit()` / `.evaluate()` / `.predict()` — and the boilerplate of training loops, batching, metric tracking, and checkpointing is handled for you.

The big shift in **Keras 3** (2023+) is that it is **backend-agnostic**: the same model code runs on **TensorFlow, PyTorch, or JAX** by flipping the `KERAS_BACKEND` environment variable. Keras went from "the friendly face of TensorFlow" to a portable modeling layer that sits on top of whichever tensor engine you (or your team) already use.

**Reach for Keras when** you want to go from idea → trained model in a handful of readable lines, you value a stable, well-documented API, or you want to write modeling code once and stay backend-portable. **Look elsewhere when** you need a fully custom, research-grade training loop with exotic control flow (raw PyTorch/JAX gives you finer control), or you're doing classical ML on tabular data (scikit-learn is the right tool — see [`scikit-learn`](scikit-learn.ipynb)).

## 2. Mental Model

Think of Keras as **LEGO for neural networks**:

- **Layers** are the bricks — each is a small function with trainable weights (`Dense`, `Conv2D`, `LSTM`, `Dropout`, ...).
- **A model** is the assembled structure — bricks snapped together so a tensor flows in one end and predictions come out the other.
- **`compile()`** bolts on the *training machinery*: a **loss** (what "wrong" means), an **optimizer** (how to adjust weights), and **metrics** (what you watch).
- **`fit()`** is the **engine** that runs the loop — forward pass → loss → backprop → weight update — over your data, epoch after epoch, so you never hand-write that loop.

```
   data ──▶ [ Input ] ─▶ [ Dense ] ─▶ [ Dropout ] ─▶ [ Dense ] ─▶ predictions
                              │                            │
                              └──────── weights ───────────┘
                                          ▲
                       compile(loss, optimizer, metrics)
                                          │
                              fit()  ──► repeat: forward, loss, backprop, update
```

Keras itself never touches the GPU directly — it lowers your layer graph onto the **backend** (TF/PyTorch/JAX), which does the actual tensor math.

## 3. Key Concepts

- **Tensor** — the n-dimensional array that flows through the network. Shapes are everything; the leading axis is almost always the **batch**.
- **Layer** — a callable object holding weights; `layer(x)` runs it and (the first time) builds its weights from the input shape.
- **Three ways to define a model:**
  - **Sequential** — a linear stack of layers; simplest, but single-input/single-output only.
  - **Functional API** — wire layers as a graph (`out = Dense(...)(x)`); handles multi-input/output, branches, shared layers.
  - **Subclassing `keras.Model`** — write `call()` yourself for full imperative control.
- **`compile(optimizer, loss, metrics)`** — declares *how* the model learns before any training happens.
- **`fit` / `evaluate` / `predict`** — train on data, score on held-out data, run inference.
- **Loss vs. metric** — the loss is what's *optimized* (differentiable); a metric is what you *report* (e.g. accuracy) and need not be differentiable.
- **Epoch / batch** — one epoch = one full pass over the data; a batch = the chunk of samples processed per weight update.
- **Callbacks** — hooks fired during `fit` (e.g. `EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau`) for control without rewriting the loop.
- **Backend** — TF / PyTorch / JAX, chosen via `KERAS_BACKEND`; the math engine under Keras 3.

## 4. Setup

Keras 3 is a thin layer over a tensor backend, so you install **Keras plus one backend**. Pick the engine your stack already uses:

```bash
pip install keras            # the Keras 3 API
pip install torch            # backend option A (PyTorch)  ← used below
# or:  pip install tensorflow      # backend option B
# or:  pip install jax jaxlib      # backend option C
```

Select the backend **before importing Keras** with the `KERAS_BACKEND` env var (or a `~/.keras/keras.json` config). The cell below pins it to PyTorch so this notebook runs anywhere torch is installed — no TensorFlow required.

In [1]:
import os
# Choose the backend BEFORE importing keras. We use torch so this runs on a
# plain CPU box with no TensorFlow install. Swap for "tensorflow" or "jax" freely.
os.environ.setdefault("KERAS_BACKEND", "torch")

import numpy as np
import keras

keras.utils.set_random_seed(42)  # reproducible weights + data shuffling
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

Keras 3.14.1 | backend: torch


## 5. Worked Examples

### Example 1 — Sequential API: a tiny binary classifier

A 2-layer MLP on a synthetic dataset where the label is a nonlinear function of 4 features. Note the rhythm every Keras model follows: **define → `compile` → `fit` → `evaluate`.** `model.summary()` prints the layer graph and parameter counts, and `fit` returns a `History` object holding per-epoch metrics.

In [2]:
# Synthetic, CPU-friendly data: label depends nonlinearly on the features.
rng = np.random.default_rng(0)
X = rng.normal(size=(400, 4)).astype("float32")
y = (X[:, 0] + X[:, 1] * X[:, 2] - X[:, 3] > 0).astype("float32")
X_train, X_test, y_train, y_test = X[:320], X[320:], y[:320], y[320:]

# define -> compile -> fit -> evaluate
model = keras.Sequential([
    keras.Input(shape=(4,)),                       # declares input shape
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),   # prob of class 1
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,   # hold out 20% of train for validation
    epochs=5, batch_size=32, verbose=2,
)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nHeld-out test accuracy: {test_acc:.3f}")
print("Final epoch val_accuracy:", round(history.history["val_accuracy"][-1], 3))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 97 (388.00 B)

 Trainable params: 97 (388.00 B)

 Non-trainable params: 0 (0.00 B)

...

Epoch 4/5


8/8 - 0s - 4ms/step - accuracy: 0.5469 - loss: 0.6678 - val_accuracy: 0.6719 - val_loss: 0.6284


Epoch 5/5


8/8 - 0s - 5ms/step - accuracy: 0.6133 - loss: 0.6538 - val_accuracy: 0.7031 - val_loss: 0.6156



Held-out test accuracy: 0.725
Final epoch val_accuracy: 0.703


### Example 2 — Functional API + save/load

The **Functional API** wires layers as an explicit graph: each layer is *called* on the previous tensor. This is what you reach for once models branch, merge, or take multiple inputs. We also show the standard persistence path — `model.save("name.keras")` writes the architecture **and** weights to a single file, and `load_model` round-trips it into an identical model.

In [3]:
# Functional API: call each layer on the previous tensor to build a graph.
inputs = keras.Input(shape=(4,), name="features")
h = keras.layers.Dense(16, activation="relu")(inputs)
h = keras.layers.Dropout(0.2)(h)          # regularization, active only during fit()
outputs = keras.layers.Dense(1, activation="sigmoid", name="prob")(h)
fmodel = keras.Model(inputs, outputs, name="functional_mlp")
fmodel.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
fmodel.fit(X_train, y_train, epochs=3, batch_size=32, verbose=0)
print("Functional model params:", fmodel.count_params())

# Save the full model (architecture + weights) and reload it.
path = "/tmp/functional_mlp.keras"
fmodel.save(path)
reloaded = keras.models.load_model(path)

before = fmodel.predict(X_test[:5], verbose=0).ravel()
after = reloaded.predict(X_test[:5], verbose=0).ravel()
print("Predictions (first 5):", np.round(before, 3))
print("Reload reproduces predictions exactly:", np.allclose(before, after))

Functional model params: 97


Predictions (first 5): [0.551 0.477 0.518 0.326 0.508]
Reload reproduces predictions exactly: True


## 6. Gotchas & Pitfalls

- **Backend must be set before `import keras`.** Changing `KERAS_BACKEND` after import has no effect — restart the kernel.
- **Input shape excludes the batch dimension.** `Input(shape=(4,))` means "4 features per sample"; Keras prepends the variable batch axis automatically. Passing `(None, 4)` or `(32, 4)` is a common shape bug.
- **Forgetting to `compile`** before `fit` raises an error; re-compiling **resets the optimizer state** (momentum, learning-rate schedules), so don't recompile mid-training unless you mean to.
- **Loss must match the output activation/label format.** `sigmoid` + `binary_crossentropy` for binary; `softmax` + `categorical_crossentropy` for one-hot labels, or `sparse_categorical_crossentropy` for integer labels. Mixing these silently trains a worse model.
- **`Dropout`/`BatchNormalization` behave differently in train vs. inference.** `fit` enables dropout; `predict`/`evaluate` disable it. If you hand-roll a loop, pass `training=True/False` yourself.
- **Unscaled features hurt convergence.** Keras won't normalize for you — standardize inputs, or add a `keras.layers.Normalization` layer adapted to your data.
- **`.keras` is the modern save format.** The old `.h5` and `SavedModel` paths still work but the single-file `.keras` zip is the recommended default in Keras 3.
- **Metrics are reported, not optimized.** Watching `accuracy` go up while `val_loss` rises is the classic overfitting signal — trust the loss for the optimization story.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs. Keras |
|---|---|---|
| **Keras 3** | Fast, readable model building; backend portability; standard training loop | Less control over exotic/custom training steps than raw frameworks |
| **Raw PyTorch** | Research, custom training loops, dynamic control flow, the dominant research ecosystem | You write the training loop, metric tracking, and checkpointing yourself |
| **PyTorch Lightning** | Structured PyTorch training without losing torch idioms | Torch-only; more concepts (LightningModule/Trainer) than Keras's `fit` |
| **JAX / Flax** | Max performance, TPUs, functional transforms (`jit`, `vmap`, `grad`) | Steeper learning curve; more boilerplate for everyday models |
| **scikit-learn** | Classical ML on tabular data (trees, SVMs, linear models) | Not for deep nets / GPUs — different problem class |
| **Hugging Face Transformers** | Pretrained LLMs/vision models and fine-tuning | Higher-level/model-zoo focused; built on torch/TF, not a from-scratch layer API |

**Rule of thumb:** prototype and ship standard architectures in **Keras**; drop to **raw PyTorch/JAX** when you need a training loop Keras's `fit` can't express. With Keras 3 you can even mix — author in Keras, run on the torch/JAX backend your team standardizes on. See also [`pytorch`](../03-llm-inference-training-optimization/pytorch.ipynb) and [`jax-flax`](jax-flax.ipynb).

## 8. Resources

- **Keras 3 documentation** — https://keras.io/
- **Developer guides (Sequential, Functional, subclassing, custom training)** — https://keras.io/guides/
- **Keras Examples (vetted, runnable recipes by task)** — https://keras.io/examples/
- **Multi-backend / backend-selection guide** — https://keras.io/getting_started/
- **KerasNLP & KerasCV (pretrained models on top of Keras 3)** — https://keras.io/keras_hub/
- **François Chollet, *Deep Learning with Python* (3rd ed.)** — the canonical book by Keras's creator: https://www.manning.com/books/deep-learning-with-python-third-edition